# AuralGuard — Kaggle Training Pipeline
Trains B1 → B2 → B3 → B5 → AuralGuard sequentially on Kaggle's free P100 GPU.

**Estimated total: ~14 hours.**
- B1 (LFCC-LCNN): ~30 min
- B2 (RawNet2): ~1 hr
- B3 (AASIST): ~2 hrs
- B5 (WavLM+AASIST): ~4 hrs
- AuralGuard (full): ~6 hrs

## 1. Setup
Downloads ASVspoof 2019 LA from Kaggle via API (requires internet).

In [ ]:
import shutil, sys
from pathlib import Path

# Find the dataset — Kaggle mounts at different paths
INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset"),
    Path("/kaggle/input/asvpoof-2019-dataset"),
    Path("/kaggle/input/asvpoof-2019-dataset/asvpoof-2019-dataset"),
]
INPUT_DIR = next((p for p in INPUT_CANDIDATES if p.exists()), None)

if INPUT_DIR is None:
    raise FileNotFoundError(
        "Add dataset via +Data button in right sidebar:\n"
        "  Search for 'awsaf49/asvpoof-2019-dataset' and click Add"
    )
print(f"Found dataset at {INPUT_DIR}")

RAW = Path("/kaggle/working/data/raw/ASVspoof2019_LA")
if not RAW.exists():
    RAW.parent.mkdir(parents=True, exist_ok=True)
    print("Copying dataset (this may take a few minutes)...")
    shutil.copytree(INPUT_DIR, RAW, dirs_exist_ok=True,
                    ignore=lambda d, files: [f for f in files if f.endswith('.zip')])
    print("Dataset ready!")
else:
    print("Dataset already prepared.")

print("Subdirs:", [p.name for p in RAW.iterdir() if p.is_dir()])

In [ ]:
# ── Install AuralGuard ──
# Clone the repo
if not Path("/kaggle/working/auralguard").exists():
    !git clone https://github.com/MIHMahmudEli/auralguard.git /kaggle/working/auralguard

%cd /kaggle/working/auralguard

# Install deps (Kaggle already has PyTorch + CUDA)
!pip install -e .[train,dev] --quiet
!pip install datasets --quiet

print("Installation complete")

In [ ]:
# ── Build manifests ──
%cd /kaggle/working/auralguard

# Fix protocol suffix for train split
import pandas as pd
from pathlib import Path

RAW = Path("/kaggle/working/data/raw/ASVspoof2019_LA")
MANIFEST_DIR = Path("/kaggle/working/data/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = ["utt_id", "path", "label", "attack", "dataset", "lang", "split", "codec"]

for split, suffix in [("train", "trn"), ("dev", "trl"), ("eval", "trl")]:
    proto = RAW / "ASVspoof2019_LA_cm_protocols" / f"ASVspoof2019.LA.cm.{split}.{suffix}.txt"
    audio_dir = RAW / f"ASVspoof2019_LA_{split}" / "flac"
    rows = []
    for line in proto.read_text().splitlines():
        parts = line.split()
        utt, attack, key = parts[1], parts[3], parts[4]
        rows.append({
            "utt_id": utt,
            "path": str(audio_dir / f"{utt}.flac"),
            "label": 0 if key == "bonafide" else 1,
            "attack": "bonafide" if key == "bonafide" else attack,
            "dataset": "asvspoof2019_la",
            "lang": "en",
            "split": split,
            "codec": "none",
        })
    df = pd.DataFrame(rows, columns=COLUMNS)
    out = MANIFEST_DIR / f"asvspoof2019_la_{split}.csv"
    df.to_csv(out, index=False)
    print(f"{out.name}: {len(df)} rows ({df.label.sum()} spoof)")

In [ ]:
# ── Build augmentation manifests (RIRs) ──
%cd /kaggle/working/auralguard

import pandas as pd
from pathlib import Path

MANIFEST_DIR = Path("/kaggle/working/data/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

# RIRs (comes with the kaggle dataset: asvpoof-2019-dataset has RIRS_NOISES)
rir_root = Path("/kaggle/working/data/raw/RIRS_NOISES")
if not rir_root.exists():
    # Try alongside LA
    for p in Path("/kaggle/working").iterdir():
        if p.is_dir() and "RIR" in p.name.upper():
            rir_root = p
            break

if rir_root.exists():
    rows = []
    for rir_dir in ["real_rirs_isotropic_noises/real_rirs_isotropic_noises",
                    "simulated_rirs/mediumroom", "simulated_rirs/largeroom",
                    "simulated_rirs/smallroom"]:
        d = rir_root / rir_dir
        if d.exists():
            for f in sorted(d.rglob("*.wav")):
                rows.append({"path": str(f)})
    pd.DataFrame(rows).to_csv(MANIFEST_DIR / "rirs.csv", index=False)
    print(f"RIRs manifest: {len(rows)} files")
else:
    print("RIRs not found, reverb augmentation disabled")

# MUSAN not available on Kaggle, add_noise will be a no-op
pd.DataFrame(columns=["path"]).to_csv(MANIFEST_DIR / "musan.csv", index=False)
print("MUSAN manifest: empty (not available)")

## 2. Train Baselines
Each cell below trains one experiment. Run them sequentially.

**Tip:** After each DONE cell, check the loss curve in TensorBoard then move to the next.

In [ ]:
# ── B1: LFCC + Light CNN (≈30 min) ──
%cd /kaggle/working/auralguard
!python scripts/train.py experiment=b1_lcnn \
    data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv \
    data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv \
    data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv
print("B1 DONE")

In [ ]:
# ── B2: RawNet2 (≈1 hr) ──
%cd /kaggle/working/auralguard
!python scripts/train.py experiment=b2_rawnet2 \
    data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv \
    data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv \
    data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv
print("B2 DONE")

In [ ]:
# ── B3: AASIST (≈2 hrs) ──
%cd /kaggle/working/auralguard
!python scripts/train.py experiment=b3_aasist \
    data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv \
    data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv \
    data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv
print("B3 DONE")

In [ ]:
# ── B5: WavLM + AASIST + OC-Softmax (≈4 hrs) ──
%cd /kaggle/working/auralguard
!python scripts/train.py experiment=b5_wavlm_ocs \
    data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv \
    data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv \
    data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv
print("B5 DONE")

In [ ]:
# ── AuralGuard (≈6 hrs) ──
%cd /kaggle/working/auralguard
!python scripts/train.py experiment=auralguard \
    data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv \
    data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv \
    data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv
print("AURALGUARD DONE!")

## 3. Evaluate & Download Results
After training, evaluate all checkpoints and download CSV results back to your machine.

In [ ]:
# ── Evaluate all experiments ──
%cd /kaggle/working/auralguard

import json, subprocess
from pathlib import Path

for exp_dir in sorted(Path("experiments").iterdir()):
    if not exp_dir.is_dir():
        continue
    ckpt = exp_dir / "checkpoints" / "best.ckpt"
    if ckpt.exists():
        out_dir = exp_dir / "eval_results"
        print(f"Evaluating {exp_dir.name}...")
        subprocess.run([
            "python", "scripts/evaluate.py",
            f"--ckpt={ckpt}",
            f"--out={out_dir}",
        ], check=True)

print("All evaluations complete!")

In [ ]:
# ── Eval all zero-shot via scripts/eval_all_zeroshot.py ──
%cd /kaggle/working/auralguard

for exp_dir in sorted(Path("experiments").iterdir()):
    if not exp_dir.is_dir():
        continue
    ckpt = exp_dir / "checkpoints" / "best.ckpt"
    if ckpt.exists():
        out_dir = exp_dir / "zeroshot_results"
        print(f"Zero-shot eval for {exp_dir.name}...")
        subprocess.run([
            "python", "scripts/eval_all_zeroshot.py",
            f"--ckpt={ckpt}",
            f"--out={out_dir}",
        ], check=True)

print("All zero-shot evaluations complete!")

## 4. Paper Figures
Generate figures for the manuscript: loss curves, EER bar chart, DET curves, score distributions.

In [ ]:
# ── Generate paper figures ──
%cd /kaggle/working/auralguard

import json, math
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"figure.dpi": 150, "font.size": 11})
FIGS = Path("/kaggle/working/paper_figures")
FIGS.mkdir(parents=True, exist_ok=True)

exp_names = sorted([p.name for p in Path("experiments").iterdir() if p.is_dir()])
model_order = ["b1_lcnn", "b2_rawnet2", "b3_aasist", "b5_wavlm_ocs", "auralguard"]
display_names = {
    "b1_lcnn": "B1 (LFCC-LCNN)",
    "b2_rawnet2": "B2 (RawNet2)",
    "b3_aasist": "B3 (AASIST)",
    "b5_wavlm_ocs": "B5 (WavLM+OCS)",
    "auralguard": "AuralGuard",
}

# ── 1. Parse TensorBoard events for loss/eer curves ──
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    colors = plt.cm.tab10(np.linspace(0, 1, len(model_order)))

    for idx, name in enumerate(model_order):
        log_dir = Path("experiments") / name / "logs"
        if not log_dir.exists():
            continue
        try:
            ea = EventAccumulator(str(log_dir))
            ea.Reload()
            tags = ea.Tags().get("scalars", [])

            for ax, tag, ylabel in zip(
                axes, ["train/loss", "eval/eer"], ["Loss", "EER (%)"]
            ):
                if tag in tags:
                    events = ea.Scalars(tag)
                    steps = [e.step for e in events]
                    vals = [e.value for e in events]
                    ax.plot(steps, vals, label=display_names.get(name, name),
                            color=colors[idx], linewidth=1.5)
                    ax.set_xlabel("Step")
                    ax.set_ylabel(ylabel)
                    ax.legend(fontsize=8, loc="upper right")
                    ax.grid(True, alpha=0.3)
        except Exception as e:
            print(f"  [skip] {name} logs: {e}")

    plt.tight_layout()
    fig.savefig(FIGS / "training_curves.png", bbox_inches="tight")
    plt.close()
    print(f"[ok] Training curves → {FIGS / 'training_curves.png'}")
except ImportError:
    print("[skip] tensorboard not installed — skipping training curves")

# ── 2. EER bar chart (in-domain + zero-shot) ──
try:
    # Collect results
    rows = []
    for name in model_order:
        res_file = Path("experiments") / name / "eval_results" / "results.json"
        if not res_file.exists():
            res_file = Path("experiments") / name / "zeroshot_results" / "results.json"
        if res_file.exists():
            data = json.loads(res_file.read_text())
            for ds_name, metrics in data.items():
                rows.append({"model": name, "dataset": ds_name, "eer": metrics["eer"]})

    if rows:
        df = pd.DataFrame(rows)
        # Keep only zero-shot datasets + in_domain_eval
        dataset_order = ["in_domain_eval", "in_the_wild", "wavefake", "mlaad",
                        "asvspoof2021_la", "asvspoof2021_df"]
        df = df[df["dataset"].isin(dataset_order)]
        df["dataset"] = pd.Categorical(df["dataset"], categories=dataset_order, ordered=True)
        df["model"] = pd.Categorical(df["model"], categories=model_order, ordered=True)
        df = df.sort_values(["dataset", "model"])

        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(dataset_order))
        n_models = len(model_order)
        bar_width = 0.15

        for i, name in enumerate(model_order):
            subset = df[df["model"] == name]
            vals = [subset[subset["dataset"] == d]["eer"].values[0]
                    if len(subset[subset["dataset"] == d]) > 0 else 0
                    for d in dataset_order]
            offset = (i - n_models / 2 + 0.5) * bar_width
            bars = ax.bar(x + offset, [v * 100 for v in vals], bar_width,
                         label=display_names.get(name, name), color=colors[i])

        ax.set_xticks(x)
        ax.set_xticklabels([d.replace("in_domain_eval", "In-Domain")
                             .replace("in_the_wild", "In-the-Wild")
                             .replace("wavefake", "WaveFake")
                             .replace("mlaad", "MLAAD")
                             .replace("asvspoof2021_la", "ASVspoof21 LA")
                             .replace("asvspoof2021_df", "ASVspoof21 DF")])
        ax.set_ylabel("EER (%)")
        ax.set_title("In-Domain & Zero-Shot EER Comparison")
        ax.legend(fontsize=8, loc="upper left")
        ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        fig.savefig(FIGS / "eer_comparison.png", bbox_inches="tight")
        plt.close()
        print(f"[ok] EER bar chart → {FIGS / 'eer_comparison.png'}")
except Exception as e:
    print(f"[skip] EER bar chart: {e}")

# ── 3. DET curves (AuralGuard vs baselines on in-domain eval) ──
try:
    from sklearn.metrics import roc_curve

    fig, ax = plt.subplots(figsize=(6, 6))

    for name, c in zip(model_order, colors):
        res_file = Path("experiments") / name / "eval_results" / "results.json"
        if not res_file.exists():
            continue
        data = json.loads(res_file.read_text())
        in_domain = data.get("in_domain_eval", {})
        if "scores" not in in_domain or "labels" not in in_domain:
            continue
        scores = np.array(in_domain["scores"])
        labels = np.array(in_domain["labels"])
        fpr, fnr, _ = roc_curve(labels, scores, pos_label=1)
        # DET: plot FRR (1 - TPR) vs FAR (FPR) on log-log scale
        far = fpr
        frr = fnr
        # Filter out zeros for log scale
        mask = (far > 1e-5) & (frr > 1e-5)
        ax.loglog(far[mask], frr[mask], label=display_names.get(name, name),
                  color=c, linewidth=1.5)

    ax.set_xlabel("False Alarm Rate (FAR)")
    ax.set_ylabel("False Rejection Rate (FRR)")
    ax.set_title("DET Curve — In-Domain Evaluation")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, which="both")
    ax.set_xlim([1e-3, 1])
    ax.set_ylim([1e-3, 1])
    plt.tight_layout()
    fig.savefig(FIGS / "det_curve.png", bbox_inches="tight")
    plt.close()
    print(f"[ok] DET curve → {FIGS / 'det_curve.png'}")
except Exception as e:
    print(f"[skip] DET curve: {e}")

# ── 4. Score distribution histogram (AuralGuard) ──
try:
    res_file = Path("experiments") / "auralguard" / "eval_results" / "results.json"
    if res_file.exists():
        data = json.loads(res_file.read_text())
        in_domain = data.get("in_domain_eval", {})
        if "scores" in in_domain and "labels" in in_domain:
            scores = np.array(in_domain["scores"])
            labels = np.array(in_domain["labels"])

            fig, ax = plt.subplots(figsize=(7, 4))
            bonafide_scores = scores[labels == 0]
            spoof_scores = scores[labels == 1]
            ax.hist(bonafide_scores, bins=80, alpha=0.6, label="Bona-fide",
                    color="green", density=True)
            ax.hist(spoof_scores, bins=80, alpha=0.6, label="Spoof",
                    color="red", density=True)
            ax.axvline(0, color="black", linestyle="--", alpha=0.5, label="Decision boundary")
            ax.set_xlabel("Spoof Score")
            ax.set_ylabel("Density")
            ax.set_title("AuralGuard — Score Distribution (In-Domain Eval)")
            ax.legend()
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            fig.savefig(FIGS / "score_distribution.png", bbox_inches="tight")
            plt.close()
            print(f"[ok] Score distribution → {FIGS / 'score_distribution.png'}")
except Exception as e:
    print(f"[skip] Score distribution: {e}")

print("All figures generated!")

In [ ]:
# ── Export results as markdown table for paper ──
%cd /kaggle/working/auralguard

rows = []
for name in model_order:
    for suffix, label in [("eval_results", "In-Domain"), ("zeroshot_results", "Zero-Shot")]:
        res_file = Path("experiments") / name / suffix / "results.json"
        if res_file.exists():
            data = json.loads(res_file.read_text())
            for ds_name, m in data.items():
                rows.append({
                    "Model": display_names.get(name, name),
                    "Dataset": ds_name,
                    "EER": f"{m['eer']:.4f}",
                    "EER CI95": f"[{m.get('eer_ci95', [0,0])[0]:.4f}, {m.get('eer_ci95', [0,0])[1]:.4f}]",
                    "AUROC": f"{m['auroc']:.4f}",
                    "min t-DCF": f"{m['min_tdcf']:.4f}",
                    "F1": f"{m.get('f1', 0):.4f}",
                    "Bal. Acc": f"{m.get('balanced_accuracy', 0):.4f}",
                })

if rows:
    import pandas as pd
    df = pd.DataFrame(rows)
    md = df.to_markdown(index=False)
    (FIGS / "results_table.md").write_text(md)
    df.to_csv(FIGS / "results_table.csv", index=False)
    print("Results table saved to paper_figures/")
    print(md)
else:
    print("No results found to tabulate")

In [ ]:
# ── Package results & figures for download ──
import shutil, tarfile
from pathlib import Path

tar_path = "/kaggle/working/auralguard_results.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    # Checkpoints
    for ckpt in Path("experiments").rglob("best.ckpt"):
        tar.add(str(ckpt), arcname=str(ckpt.relative_to("/kaggle/working")))
    # Evaluation results
    for res in Path("experiments").rglob("results.json"):
        tar.add(str(res), arcname=str(res.relative_to("/kaggle/working")))
    # Figures
    for fig in FIGS.glob("*"):
        tar.add(str(fig), arcname=f"paper_figures/{fig.name}")

print(f"Results packaged: {tar_path}")
print(f"  Size: {Path(tar_path).stat().st_size / 1e6:.1f} MB")
print("Download via Kaggle sidebar → Output → Data")